# Linear Regression

The foundation of supervised learning. This notebook covers:

1. **Simple & Multiple Linear Regression** - OLS, assumptions, interpretation
2. **Regularization** - Ridge (L2), Lasso (L1), ElasticNet
3. **Polynomial Regression** - Capturing nonlinear relationships
4. **End-to-End Pipeline** with proper validation

**Dataset**: California Housing - predict median house value

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set_theme(style="whitegrid")

In [ ]:
# Load data
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f"Shape: {df.shape}")
print(f"\nTarget (MedHouseVal): median house value in $100k")
df.head()

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 1. Ordinary Least Squares (OLS)

Minimizes: $\sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = ||\mathbf{y} - \mathbf{X}\boldsymbol{\beta}||^2_2$

Closed-form solution: $\boldsymbol{\hat{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$

In [ ]:
# OLS Linear Regression
ols_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])
ols_pipe.fit(X_train, y_train)
y_pred_ols = ols_pipe.predict(X_test)

print("OLS Linear Regression")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ols)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_ols):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_ols):.4f}")

# Coefficient interpretation
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": ols_pipe.named_steps["regressor"].coef_
}).sort_values("Coefficient", key=abs, ascending=False)

print(f"\nIntercept: {ols_pipe.named_steps['regressor'].intercept_:.4f}")
print("\nStandardized Coefficients (feature importance):")
print(coef_df.to_string(index=False))

## 2. Regularized Regression

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | $\lambda \sum \beta_j^2$ | Shrinks coefficients toward zero |
| **Lasso (L1)** | $\lambda \sum |\beta_j|$ | Drives some coefficients exactly to zero (feature selection) |
| **ElasticNet** | $\lambda_1 \sum |\beta_j| + \lambda_2 \sum \beta_j^2$ | Combines L1 and L2 |

In [ ]:
# Compare regularization methods
results = {}

models = {
    "OLS": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 50)),
    "Lasso": LassoCV(alphas=np.logspace(-3, 3, 50), max_iter=10000, cv=5),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
}

for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("reg", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    results[name] = {
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred),
        "Non-zero coefs": np.sum(pipe.named_steps["reg"].coef_ != 0),
    }

results_df = pd.DataFrame(results).T
print(results_df.round(4))

In [ ]:
# Visualize coefficient paths as regularization strength increases
alphas = np.logspace(-2, 4, 100)
ridge_coefs = []
lasso_coefs = []

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_s, y_train)
    ridge_coefs.append(ridge.coef_)
    
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_s, y_train)
    lasso_coefs.append(lasso.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(X.columns):
    axes[0].plot(alphas, [c[i] for c in ridge_coefs], label=name)
    axes[1].plot(alphas, [c[i] for c in lasso_coefs], label=name)

for ax, title in zip(axes, ["Ridge (L2)", "Lasso (L1)"]):
    ax.set_xscale("log")
    ax.set_xlabel("Alpha (regularization strength)")
    ax.set_ylabel("Coefficient Value")
    ax.set_title(f"{title} Coefficient Paths")
    ax.legend(fontsize=7)
    ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_ols, alpha=0.2, s=10, color="teal")
ax.plot([0, 5], [0, 5], "r--", lw=2)
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title("OLS: Predicted vs Actual")
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Always scale features** for regularized regression - penalties depend on coefficient magnitude
2. **Ridge** is the default choice - robust, keeps all features, handles multicollinearity
3. **Lasso** for feature selection - automatically zeros out irrelevant features
4. **ElasticNet** combines both - useful when features are correlated
5. **Use CV variants** (RidgeCV, LassoCV) to automatically select the best regularization strength